# Train on 200 Extracted Videos

**Data:** 200 videos, 46K utterances, 0.5% positive
**Problem:** Severe class imbalance - need strong class weights

**Strategy:**
1. Use strong class weights [1.0, 100.0] for 200:1 imbalance
2. Train MLP fusion on 21-dim prosody
3. If F1 > 0.5, expand to full 555 videos

In [ ]:
# @title Step 1: Mount Drive
from google.colab import drive
drive.mount('/content/gdrive')
print('Drive mounted!')

In [ ]:
# @title Step 2: Load checkpoint
import json
import numpy as np
from pathlib import Path

CHECKPOINT = Path('/content/gdrive/MyDrive/prosody_555_checkpoint.json')

print('Loading checkpoint...')
with open(CHECKPOINT) as f:
    data = json.load(f)

prosody_data = data['prosody_data']
processed = set(data['processed_videos'])
failed = set(data['failed_videos'])

print(f"Videos: {len(prosody_data)}")
print(f"Failed: {len(failed)}")

In [ ]:
# @title Step 3: Prepare data
from collections import defaultdict

# Collect all data
all_X = []
all_y = []

for vid, utts in prosody_data.items():
    for utt in utts:
        all_X.append(utt['prosody'])
        all_y.append(utt['label'])

X = np.array(all_X, dtype=np.float32)
y = np.array(all_y, dtype=np.int64)

print(f"Total: {len(X)} samples")
print(f"Positive: {y.sum()} ({y.mean()*100:.1f}%)")
print(f"Shape: {X.shape}")

In [ ]:
# @title Step 4: Split by video (no leakage)
from sklearn.model_selection import train_test_split

# Group by video
video_groups = defaultdict(list)
for vid, utts in prosody_data.items():
    for i, utt in enumerate(utts):
        video_groups[vid].append((utt['prosody'], utt['label']))

videos = list(video_groups.keys())
np.random.seed(42)
np.random.shuffle(videos)

# 80/10/10 split
n_train = int(0.8 * len(videos))
n_val = int(0.1 * len(videos))

train_vids = set(videos[:n_train])
val_vids = set(videos[n_train:n_train+n_val])
test_vids = set(videos[n_train+n_val:])

X_train, y_train = [], []
X_val, y_val = [], []
X_test, y_test = [], []

for vid, utts in prosody_data.items():
    for utt in utts:
        x = utt['prosody']
        label = utt['label']
        if vid in train_vids:
            X_train.append(x)
            y_train.append(label)
        elif vid in val_vids:
            X_val.append(x)
            y_val.append(label)
        else:
            X_test.append(x)
            y_test.append(label)

X_train = np.array(X_train, dtype=np.float32)
y_train = np.array(y_train, dtype=np.int64)
X_val = np.array(X_val, dtype=np.float32)
y_val = np.array(y_val, dtype=np.int64)
X_test = np.array(X_test, dtype=np.float32)
y_test = np.array(y_test, dtype=np.int64)

print(f"Train: {len(X_train)} ({y_train.sum()} pos, {y_train.mean()*100:.2f}%)")
print(f"Val: {len(X_val)} ({y_val.sum()} pos, {y_val.mean()*100:.2f}%)")
print(f"Test: {len(X_test)} ({y_test.sum()} pos, {y_test.mean()*100:.2f}%)")

In [ ]:
# @title Step 5: Train MLP with strong class weights
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

class ProsodyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Calculate class weights - need VERY strong weight for minority
pos_count = y_train.sum()
neg_count = len(y_train) - pos_count
pos_weight = len(y_train) / (2 * pos_count + 1e-8)
print(f"Pos weight: {pos_weight:.1f}")

train_ds = ProsodyDataset(X_train, y_train)
val_ds = ProsodyDataset(X_val, y_val)
test_ds = ProsodyDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)
test_loader = DataLoader(test_ds, batch_size=256)

model = nn.Sequential(
    nn.Linear(21, 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

EPOCHS = 20
best_f1 = 0
best_state = None

for epoch in range(EPOCHS):
    t0 = time.time()
    
    # Train
    model.train()
    train_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x).squeeze(-1)
        loss = criterion(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    
    scheduler.step()
    
    # Validate
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            out = model(x).squeeze(-1)
            val_preds.extend((torch.sigmoid(out) > 0.5).cpu().numpy())
            val_labels.extend(y.numpy())
    
    val_f1 = f1_score(val_labels, val_preds)
    epoch_time = time.time() - t0
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f} | Time: {epoch_time:.1f}s")
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"  New best!")

In [ ]:
# @title Step 6: Evaluate on test set
model.load_state_dict(best_state)
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        out = model(x).squeeze(-1)
        test_preds.extend((torch.sigmoid(out) > 0.5).cpu().numpy())
        test_labels.extend(y.numpy())

test_f1 = f1_score(test_labels, test_preds)
print(f"\n=== TEST RESULTS ===")
print(f"Test F1: {test_f1:.4f}")
print()
print(classification_report(test_labels, test_preds, target_names=['No Laughter', 'Laughter']))

# Save model
torch.save(best_state, '/content/gdrive/MyDrive/prosody_model_200.pt')
print("Model saved!")